# All of India: Earth Engine export, then forecasts with and without non-forest factors

**Runtime:** a normal CPU runtime is enough (Runtime → Change runtime type → CPU). Earth Engine does the heavy work on Google's side.

**Use one Google account for everything** (Colab, Drive, Earth Engine).

## One-time setup (10 minutes, only the first time)
1. Open https://code.earthengine.google.com/register while signed in to that account.
2. Choose **Unpaid usage → Noncommercial**, project type **Academia & research** (you are a student doing research).
3. Choose **Create a new Google Cloud project**. Give it an ID such as `canopy-watch-india`, accept the terms. If asked for a tier, pick the free one.
4. Copy the project ID; you paste it in cell 2 below.

You do not need the Earth Engine Code Editor or any JavaScript. Everything happens in this notebook.

## 1. Drive and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/deforestation-india'
!mkdir -p $WORK
%cd /content
!rm -rf deforestation-prediction-india
!git clone -q https://github.com/kavya-sajjit/deforestation-prediction-india.git
%cd deforestation-prediction-india
!pip install -q rasterio earthengine-api

## 2. Connect to Earth Engine
A Google sign-in window opens: pick the same account and allow access. Then check the list it prints: the country names should be India plus the Kashmir and Arunachal units GAUL keeps separately.

In [ ]:
PROJECT = 'canopy-watch-india'   # <- your Cloud project ID from the setup steps
import ee
ee.Authenticate()
ee.Initialize(project=PROJECT)
from deforest.national import gee
gee.print_units()

## 3. Start the exports (runs on Earth Engine, not here)
This queues four jobs and returns in seconds. They usually take one to a few hours. **You can close this notebook**; they keep running and write into the Drive folder `deforestation-india`.

In [ ]:
tasks = gee.start_exports()

## 4. Check progress
Run this whenever you like. If Colab restarted, run cells 1 and 2 again, then this cell still works. All four must say COMPLETED before step 5. If one says FAILED, copy its message and send it to me.

In [ ]:
gee.recent()

## 5. Stitch the exports into one grid (a few minutes)

In [ ]:
from deforest.national import table
info = table.build(src=WORK, out=f'{WORK}/india_grid.npz')

## 6. Models, projection and district totals
Trains the three models once per state (leave-one-state-out). Expect 30-90 minutes on a CPU runtime. Results are written to `deforestation-india/results`.

In [ ]:
from deforest.national import model
from IPython.display import Markdown, display
text = model.run(f'{WORK}/india_grid.npz', f'{WORK}/india_admin_codes.csv', f'{WORK}/results')
display(Markdown(text))

## 7. Bring the results to the repo
Download from Drive `deforestation-india/results/` the files `NATIONAL.md`, `loso.csv`, `importance.csv`, `projection_backtest.csv`, `districts.json`, plus `deforestation-india/india_districts.geojson`. Put them in `results/national/` in the repo and commit. The risk file (`risk_2026_2029.csv.gz`) is large; keep it on Drive.